In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# Define source file path, target table name, and natural key
volume_path = "/Volumes/bronze_dev/opportunity_insights/opportunity_insights_raw"
file_path_csv = f"{volume_path}/social_capital_zipcode.csv"
url = "https://data.humdata.org/dataset/85ee8e10-0c66-4635-b997-79b6fad44c71/resource/ab878625-279b-4bef-a2b3-c132168d536e/download/social_capital_zip.csv"
table_name = "bronze_dev.opportunity_insights.social_capital_zip"
natural_key = ["zip"]

Copy the csv from the remote URL to the local volume

In [0]:
dbutils.fs.cp(url, file_path_csv)

Display contents of the volume

In [0]:
display(dbutils.fs.ls(volume_path))

Prep table and perform upsert

In [0]:
import src.utils.helpers

# Read source file
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(file_path_csv)

df_prepped = src.utils.helpers.prep_bronze_df(df)
src.utils.helpers.upsert_table(table_name, df_prepped, natural_key, spark)

Verify new table

In [0]:
%sql
SELECT * FROM bronze_dev.opportunity_insights.social_capital_zip